# 第10章 学習のお医者さん：うまく学習させる技

初期化・正規化・Adam・ドロップアウト・学習率スケジュールを救急箱に

教科書のページ：https://sakunokyumei.github.io/llm-research-textbook/10-training-tips.html

上のセルから順に、再生ボタン（▶）か `Shift + Enter` で実行します。説明は教科書のページにあります。


## 準備


In [ ]:
import re, math, urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F

def load_soseki(file_id="773_ruby_5968"):
    url = ("https://raw.githubusercontent.com/aozorahack/aozorabunko_text/master/"
           f"cards/000148/files/{file_id}/{file_id}.txt")
    text = urllib.request.urlopen(url).read().decode("cp932")
    text = text.split("-------------------------------------------------------")[2]
    text = text.split("底本：")[0]
    text = re.sub(r"《.*?》", "", text)
    text = re.sub(r"※?［＃.*?］", "", text)
    text = text.replace("｜", "").replace("\r\n", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

text = load_soseki()
chars = sorted(set(text))
V = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text])
n = int(len(data) * 0.9)
train_data, val_data = data[:n], data[n:]
T = 8

class MLP(nn.Module):
    def __init__(self, V, T, d=32, H=256, dropout=0.0):
        super().__init__()
        self.emb = nn.Embedding(V, d)
        self.fc1 = nn.Linear(T * d, H)
        self.ln = nn.LayerNorm(H)           # この章で学ぶ「層正規化」
        self.drop = nn.Dropout(dropout)     # この章で学ぶ「ドロップアウト」
        self.fc2 = nn.Linear(H, V)

    def forward(self, x):
        e = self.emb(x)
        h = torch.tanh(self.ln(self.fc1(e.view(len(x), -1))))
        return self.fc2(self.drop(h))

val_X = torch.stack([val_data[i:i + T] for i in range(len(val_data) - T)])
val_Y = val_data[T:]

@torch.no_grad()
def val_loss(model):
    model.eval()
    loss = F.cross_entropy(model(val_X), val_Y).item()
    model.train()
    return loss

## 診察1：最初の損失を見る


In [ ]:
torch.manual_seed(0)
model = MLP(V, T)
x, y = val_X[:512], val_Y[:512]
print("理想の最初の損失 log V：", round(math.log(V), 3))
print("このモデルの最初の損失：", round(F.cross_entropy(model(x), y).item(), 3))

with torch.no_grad():
    model.fc2.weight *= 30          # わざと出力層の重みを30倍にしてみる
print("重みを30倍にしたら：", round(F.cross_entropy(model(x), y).item(), 3))

## 統計の道具：平均・分散・標準偏差


In [ ]:
z = torch.randn(10000)       # 「標準正規分布」から10000個の数を引く
print("平均：", round(z.mean().item(), 3))
print("分散：", round(z.var().item(), 3))
print("標準偏差：", round(z.std().item(), 3))

In [ ]:
!pip install -q matplotlib-fontja

In [ ]:
import matplotlib.pyplot as plt
import matplotlib_fontja

plt.figure(figsize=(6, 3))
plt.hist(z.numpy(), bins=60)
plt.title("torch.randn の数のばらつき（正規分布）")
plt.xlabel("値"); plt.ylabel("個数")
plt.show()

## 診察2：層を通るたびに暴れる数


In [ ]:
def stack_stats(weight_scale, layers=10, width=256):
    torch.manual_seed(0)
    h = torch.randn(1000, width)
    stds = []
    for _ in range(layers):
        W = torch.randn(width, width) * weight_scale
        h = torch.tanh(h @ W)
        stds.append(round(h.std().item(), 3))
    return stds

print("重みの大きさ 1.0  ：", stack_stats(1.0))
print("重みの大きさ 0.01 ：", stack_stats(0.01))
print("重みの大きさ 1/√256：", stack_stats(1 / math.sqrt(256)))

## 治療1：LayerNorm（層正規化）


In [ ]:
h = torch.randn(2, 6) * 5 + 3      # 平均3、標準偏差5くらいの、暴れた数
mean = h.mean(dim=1, keepdim=True)
std = h.std(dim=1, keepdim=True, unbiased=False)
h_hand = (h - mean) / (std + 1e-5)  # 平均を引いて、標準偏差で割る

ln = nn.LayerNorm(6, elementwise_affine=False)
print(h_hand)
print(ln(h))

## 治療2：Adam（賢い坂の下り方）


In [ ]:
def train(model, steps=6000, lr=1e-3, weight_decay=0.0, schedule=False, seed=0):
    g = torch.Generator().manual_seed(seed)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = []
    for step in range(steps + 1):
        if schedule:                                   # 学習率スケジュール（後で説明）
            for group in opt.param_groups:
                group["lr"] = lr_at(step, lr, steps)
        ix = torch.randint(0, len(train_data) - T, (256,), generator=g)
        x = torch.stack([train_data[i:i + T] for i in ix])
        y = train_data[ix + T]
        loss = F.cross_entropy(model(x), y)
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # 勾配クリッピング（後で説明）
        opt.step()
        if step % 1000 == 0:
            history.append((step, loss.item(), val_loss(model)))
            print(f"ステップ {step:4d}：訓練 {history[-1][1]:.3f}  検証 {history[-1][2]:.3f}")
    return history

torch.manual_seed(0)
model_a = MLP(V, T)
hist_a = train(model_a, lr=1e-3)

## 診察3：過学習


In [ ]:
def plot_history(hists, labels, title):
    plt.figure(figsize=(6, 3.5))
    for h, label in zip(hists, labels):
        s = [r[0] for r in h]
        plt.plot(s, [r[1] for r in h], linestyle="--", label=label + "（訓練）")
        plt.plot(s, [r[2] for r in h], label=label + "（検証）")
    plt.ylim(1.5, 4.5)
    plt.xlabel("ステップ"); plt.ylabel("損失")
    plt.legend(); plt.title(title)
    plt.show()

plot_history([hist_a], ["AdamW"], "訓練は下がり続け、検証は途中から悪化")

## 治療4：学習率スケジュール


In [ ]:
def lr_at(step, max_lr, total, warmup=200):
    if step < warmup:
        return max_lr * step / warmup                       # ウォームアップ：0から徐々に上げる
    progress = (step - warmup) / (total - warmup)
    return max_lr * 0.5 * (1 + math.cos(math.pi * progress))  # コサイン減衰：なめらかに0へ

steps = list(range(6001))
plt.figure(figsize=(6, 2.8))
plt.plot(steps, [lr_at(s, 2e-3, 6000) for s in steps])
plt.xlabel("ステップ"); plt.ylabel("学習率")
plt.title("ウォームアップ＋コサイン減衰")
plt.show()

## 全部の治療を施す


In [ ]:
torch.manual_seed(0)
model_b = MLP(V, T, dropout=0.3)
hist_b = train(model_b, lr=2e-3, weight_decay=0.1, schedule=True)

In [ ]:
plot_history([hist_a, hist_b], ["治療前", "治療後"], "治療前と治療後")

## 学習したモデルを保存する


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
save_dir = "/content/drive/MyDrive/tsukuru-llm"
os.makedirs(save_dir, exist_ok=True)
torch.save(model_b.state_dict(), f"{save_dir}/mlp_kokoro.pt")
print("保存しました：", os.listdir(save_dir))

In [ ]:
loaded = MLP(V, T, dropout=0.3)
loaded.load_state_dict(torch.load(f"{save_dir}/mlp_kokoro.pt"))
print("読み込んだモデルの検証損失：", round(val_loss(loaded), 3))